# Проблемы с оберткой `aizynthfinder`

Оригинальный `aizynthfinder.aizynthfinder.AiZynthFinder` - по сути своей, набор функций вокруг базового класса `aizynthfinder.context.config.Configuration`. Этот класс собирается при чтении файла или сборки из словаря и передаётся целиком в каждый компонент `AiZynthFinder`. К сожалению, внутри `Configuration` содержатся не только неизменяемые объекты, но и списки, используемых в данный момент, стоков, скореров и тд, а также их кэш.

Чтобы параллельные потоки не создавали гонку и не искажали вычисления друг друга, нужно превратить `AiZynthFinder` В чистую функцию, без сайд-эффектов.


In [1]:
from chemrar_retro import retro, config, scorers
from pydantic import BaseModel
import pandas as pd
import numpy as np

Чтобы проверить наличие сайд-эффектов был написан класс `Snapshot`, который рекурсивно делает снимок всех внутренних компонентов до определённого уровня. (Чтобы проверка была быстрой, он не проверяет значения, если в списке или в словаре оказывается слишком много данных)


In [2]:
# snapshot
from typing import Any, Iterator
import pickle
from pathlib import Path
from typing import Hashable


class Snapshot:
    def __init__(self, obj, *, depth: int, file: Path | None = None) -> None:
        self.depth = depth
        self.dump_file = file

        if file:
            with open(file.as_posix(), "wb") as f:
                pickle.dump(
                    Snapshot._to_image(obj, depth),
                    file=f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )
            self.dump = None
        else:
            self.dump = pickle.dumps(
                Snapshot._to_image(obj, depth),
                protocol=pickle.HIGHEST_PROTOCOL,
            )

    def get_image(self):
        return self._get_image()

    def diff(self, obj, max_depth: int | None = None) -> dict[str, tuple]:
        depth = min(self.depth, max_depth or self.depth)
        a = self._get_image()

        diff = Snapshot._snapshot_diff(a, Snapshot._to_image(obj, max_depth=self.depth))
        result = {}
        for i, x0, x1 in diff:
            title = []

            for t in i.replace("]", "").split("[")[: depth + 1]:
                if t == "":
                    continue
                if t.startswith("'."):
                    title.append(t.replace("'", ""))
                else:
                    title.append(f"[{t}]")

            key = "".join(title).strip()

            # key = "".join(
            #                 f"['{t}']" if t and not t.startswith(".") else t

            #             ).strip()

            result.update({key: (x0, x1)})

        return result

    def _get_image(self) -> dict:  # type: ignore
        if self.dump_file:
            with open(self.dump_file, "rb") as f:
                return pickle.load(f)
        elif self.dump:
            return pickle.loads(self.dump)

    @staticmethod
    def _to_image(
        obj: object,
        max_depth: int,
        depth: int = 0,
        memo: frozenset[int] = frozenset(),
    ) -> object:

        if depth > max_depth:
            return _DepthLimited(type(obj).__name__)

        if id(obj) in memo:
            return _Skipped("cycle")

        if isinstance(obj, dict):
            memo = memo | {id(obj)}
            if len(obj) > 500:
                return _Skipped("long dict")

            return {
                k if isinstance(k, Hashable) else "@nonHashable@": Snapshot._to_image(
                    v, max_depth, depth + 1, memo
                )
                if isinstance(k, str)
                else (
                    Snapshot._to_image(k, max_depth, depth + 1, memo),
                    Snapshot._to_image(v, max_depth, depth + 1, memo),
                )
                for k, v in obj.items()
            }

        if isinstance(obj, (list, tuple, set)):
            memo = memo | {id(obj)}
            if len(obj) > 500:

                return _Skipped("long sequence")
            return [Snapshot._to_image(v, max_depth, depth + 1, memo) for v in obj]

        if isinstance(obj, (pd.DataFrame, pd.Series, np.ndarray)):
            return _Skipped("data")


        attrs: dict[str, object] = {}
        if hasattr(obj, "__dict__"):
            attrs.update(vars(obj))
        for name in getattr(type(obj), "__slots__", ()):
            if hasattr(obj, name):
                attrs[name] = getattr(obj, name)

        if attrs:
            return {
                "__type__": type(obj).__name__,
                **{
                    f".{k}": Snapshot._to_image(v, max_depth, depth + 1, memo)
                    for k, v in attrs.items()
                },
            }

        return repr(obj)

    @staticmethod
    def _snapshot_diff(a: Any, b: Any, path: str = "") -> Iterator[tuple[str, Any, Any]]:
        if type(a) is not type(b):
            yield path, a, b
            return

        if isinstance(a, dict):
            for key in a.keys() | b.keys():
                if key not in a:
                    yield f"{path}[{key!r}]", "<missing>", b[key]
                elif key not in b:
                    yield f"{path}[{key!r}]", a[key], "<missing>"
                else:
                    yield from Snapshot._snapshot_diff(a[key], b[key], f"{path}[{key!r}]")
            return

        if isinstance(a, (list, tuple)):
            if len(a) != len(b):
                yield f"{path}.len", len(a), len(b)
                return
            for i, (av, bv) in enumerate(zip(a, b)):
                yield from Snapshot._snapshot_diff(av, bv, f"{path}[{i}]")
            return

        if isinstance(a, set):
            if a != b:
                yield path, a - b, b - a
            return
        try:
            if a != b:
                yield path, a, b
        except:
            if all(a != b):
                yield path, a, b


class _Skipped:
    __slots__ = ("type_name",)

    def __init__(self, type_name: str) -> None:
        self.type_name = type_name

    def __repr__(self) -> str:
        return f"<skipped:{self.type_name}>"

    def __hash__(self) -> int:
        return hash(self.type_name + "_Skipped")

    def __eq__(self, other: object) -> bool:
        return isinstance(other, _Skipped) and self.type_name == other.type_name


class _DepthLimited:
    __slots__ = ("type_name",)

    def __init__(self, type_name: str) -> None:
        self.type_name = type_name

    def __repr__(self) -> str:
        return f"<depth-limit:{self.type_name}>"

    def __hash__(self) -> int:
        return hash(self.type_name + "_DepthLimited")

    def __eq__(self, other: object) -> bool:
        return isinstance(other, _DepthLimited) and self.type_name == other.type_name


In [ ]:
class A(BaseModel):
    a: int = 1
    b: dict = dict(x=0, y=8)
    c: list = [1, 2]


a1 = A()
a2 = A(
    a=1,
    b=dict(x=0, y=7),
    c=[1, 3],
)

s1 = Snapshot(a1, depth=10)
diff = s1.diff(a2)

print("\n".join(f"{e:3}|{i}" for e, i in enumerate(diff)))

In [6]:
engine = retro.create_engine(
    expansion=dict(
        uspto=config.ExpansionPolicy(
            model=Path("data/model/uspto_model.onnx"),
            template=Path("data/model/uspto_templates.csv.gz"),
        ),
        ringbreaker=config.ExpansionPolicy(
            model=Path("data/model/uspto_ringbreaker_model.onnx"),
            template=Path("data/model/uspto_ringbreaker_templates.csv.gz"),
        ),
    ),
    filter=dict(
        uspto=config.FilterPolicy(model=Path("data/model/uspto_filter_model.onnx")),
    ),
    stock=dict(
        zinc=config.Stock(path=Path("data/model/zinc_stock.hdf5")),
    ),
    scorers=[(scorers.availability. FractionInStockScorer, {})],
)

smiles = "Cc1cccc(C)c1N(CC(=O)Nc1ccc(-c2ncon2)cc1)C(=O)C1CCS(=O)(=O)CC1"

Предельный уровень копирования =8. Дальше внутри библиотеки находится `Lock`,


In [7]:
snap = Snapshot(engine,depth=8)

# Тест функции копирования


Решение проблемы гонки - скопировать все лёгкие компоненты, которые меняются при применении. Это делает функция `retro._copy_engine`, она просто копирует все внутренние части стоков, скореры, стратегии фильтрации и расширения вместе с их кешами, но при этом не трогает веса моделей.


In [11]:
def imutate_func(
    engine: retro.Engine,
    smiles: str,
):
    engine = retro._copy_engine(engine)

    engine.target_smiles = smiles
    engine.expansion_policy.select_all()
    engine.tree_search()
    engine.build_routes()
    return

In [12]:
imutate_func(engine,smiles=smiles )

В основные компоненты не входит логер. Конечно, его стоит скопировать, но для задачи тестового задания этого достаточно.


In [14]:
diff = snap.diff(engine)
diff_keys = [k for k in diff if "logger" not in k]

print("\n".join(f"{e:3}|{i}" for e, i in enumerate(diff_keys)))

In [17]:
diff.keys()

dict_keys(['.config.scorers._config.scorers._config.scorers._logger._cache', '.config.scorers._config.scorers._config.scorers._logger.level', '.config.scorers._config.scorers._config._logger._cache[10]', '.config.scorers._config.scorers._config._logger._cache[20]', '.config.scorers._config.scorers._config._logger.manager.loggerDict', '.config.scorers._config.scorers._config._logger.level', '.config.scorers._config.scorers._config.expansion_policy._logger._cache', '.config.scorers._config.scorers._config.expansion_policy._logger.level', '.config.scorers._config.scorers._config.filter_policy._logger._cache', '.config.scorers._config.scorers._config.filter_policy._logger.level', '.config.scorers._config.scorers._config.stock._logger._cache', '.config.scorers._config.scorers._config.stock._logger.level', '.config.scorers._config.scorers._logger._cache[10]', '.config.scorers._config.scorers._logger._cache[20]', ".config.scorers._config.scorers._logger.manager.loggerDict['parso.cache']", ".c

# Тест работы
